# Build a Cortex Agent from Scratch with Snowflake

This notebook walks you through building a pipeline that lets users ask questions in plain English and get answers from both **structured data** (tables) and **unstructured data** (documents) stored in Snowflake.

The pipeline has two paths:
- **Cortex Analyst + Semantic Views** -- translates natural language into SQL for structured data
- **Cortex Search** -- retrieves relevant documents via semantic similarity for unstructured data

**What you'll build:**
- Sample sales, inventory, and product documentation tables
- A **Semantic View** that teaches Cortex Analyst about your data's dimensions and metrics
- A **Cortex Search Service** that indexes documents for retrieval-augmented generation (RAG)


**Prerequisites:**
- Snowflake account with Cortex AI enabled
- ACCOUNTADMIN role (or equivalent privileges)
- A running warehouse (COMPUTE_WH)


---
## 1. Setup & Prerequisites

**Objective:** Set up the environment and verify Cortex AI access.

### Step 1.1: Set Up Environment

We create a dedicated database and schema for this tutorial. Using `ACCOUNTADMIN` ensures we have the necessary privileges to create semantic views and search services.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
CREATE DATABASE IF NOT EXISTS CORTEX_AGENTS_LAB;
USE DATABASE CORTEX_AGENTS_LAB;
CREATE SCHEMA IF NOT EXISTS TUTORIAL;
USE SCHEMA TUTORIAL;

### Step 1.2: Verify Cortex Access

This quick test confirms your account has Cortex AI enabled. If this fails, Cortex may not be available in your region or account.

In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', 'Say hello in one word') AS cortex_test;

---
## 2. Create Sample Data

**Objective:** Load structured and unstructured data that Cortex AI will query via natural language.

We create three tables:
- **sales** -- transaction-level sales data (structured, queried via Cortex Analyst)
- **inventory** -- current stock levels (structured, queryable via custom procedures)
- **product_docs** -- product documentation (unstructured text, indexed by Cortex Search)

### Step 2.1: Create Sales Table

This is our primary structured dataset. Cortex Analyst will generate SQL against this table using the semantic view we build in Section 3.

In [ ]:
CREATE OR REPLACE TABLE sales (
    sale_id NUMBER AUTOINCREMENT,
    sale_date DATE,
    product_name VARCHAR,
    category VARCHAR,
    region VARCHAR,
    quantity NUMBER,
    unit_price NUMBER(10,2),
    total_amount NUMBER(10,2),
    customer_segment VARCHAR
);

### Step 2.2: Load Sample Sales Data

12 rows of sales transactions across 3 regions, 2 categories, and 3 customer segments.

In [ ]:
INSERT INTO sales (sale_date, product_name, category, region, quantity, unit_price, total_amount, customer_segment)
VALUES
    ('2024-01-15', 'Laptop Pro', 'Electronics', 'North America', 10, 1299.99, 12999.90, 'Enterprise'),
    ('2024-01-16', 'Wireless Mouse', 'Electronics', 'Europe', 50, 29.99, 1499.50, 'SMB'),
    ('2024-01-17', 'Office Chair', 'Furniture', 'North America', 20, 299.99, 5999.80, 'Enterprise'),
    ('2024-01-18', 'Standing Desk', 'Furniture', 'Asia Pacific', 15, 499.99, 7499.85, 'SMB'),
    ('2024-01-19', 'Monitor 27"', 'Electronics', 'Europe', 30, 399.99, 11999.70, 'Enterprise'),
    ('2024-01-20', 'Keyboard Pro', 'Electronics', 'North America', 100, 149.99, 14999.00, 'Consumer'),
    ('2024-02-01', 'Laptop Pro', 'Electronics', 'Asia Pacific', 25, 1299.99, 32499.75, 'Enterprise'),
    ('2024-02-05', 'Webcam HD', 'Electronics', 'North America', 75, 79.99, 5999.25, 'SMB'),
    ('2024-02-10', 'Office Chair', 'Furniture', 'Europe', 40, 299.99, 11999.60, 'Enterprise'),
    ('2024-02-15', 'Headphones', 'Electronics', 'North America', 60, 199.99, 11999.40, 'Consumer'),
    ('2024-03-01', 'Standing Desk', 'Furniture', 'North America', 35, 499.99, 17499.65, 'Enterprise'),
    ('2024-03-10', 'Laptop Pro', 'Electronics', 'Europe', 20, 1299.99, 25999.80, 'SMB');

### Step 2.3: Create Inventory Table

Stock levels for each product. This data can be queried directly or exposed through custom stored procedures in downstream applications.

In [ ]:
CREATE OR REPLACE TABLE inventory (
    product_name VARCHAR,
    sku VARCHAR,
    quantity_in_stock NUMBER,
    reorder_level NUMBER,
    unit_cost NUMBER(10,2),
    last_restocked DATE
);

In [ ]:
INSERT INTO inventory VALUES
    ('Laptop Pro', 'LP-001', 45, 20, 899.99, '2024-03-01'),
    ('Wireless Mouse', 'WM-002', 500, 100, 12.99, '2024-02-15'),
    ('Office Chair', 'OC-003', 75, 25, 149.99, '2024-02-20'),
    ('Standing Desk', 'SD-004', 30, 15, 299.99, '2024-03-05'),
    ('Monitor 27"', 'MN-005', 60, 20, 249.99, '2024-02-28'),
    ('Keyboard Pro', 'KP-006', 200, 50, 79.99, '2024-03-10');

### Step 2.4: Create Product Documentation Table

Unstructured text content -- specs, troubleshooting guides, assembly instructions. Cortex Search will index the `content` column for semantic retrieval.

In [ ]:
CREATE OR REPLACE TABLE product_docs (
    doc_id NUMBER AUTOINCREMENT,
    product_name VARCHAR,
    doc_type VARCHAR,
    content VARCHAR,
    last_updated DATE
);

In [ ]:
INSERT INTO product_docs (product_name, doc_type, content, last_updated)
VALUES
    ('Laptop Pro', 'specifications', 
     'The Laptop Pro features a 15.6-inch 4K display, Intel i9 processor, 32GB RAM, and 1TB SSD. Battery life is up to 12 hours. Includes Thunderbolt 4 ports and Wi-Fi 6E. Weight: 4.2 lbs. Warranty: 3 years standard, extendable to 5 years.',
     '2024-01-01'),
    ('Laptop Pro', 'troubleshooting',
     'Common issues: 1) Battery drain - check background apps and reduce screen brightness. 2) Overheating - ensure vents are not blocked, use on hard surface. 3) Slow performance - check for updates, run disk cleanup. 4) Wi-Fi issues - update network drivers, reset network settings.',
     '2024-01-15'),
    ('Standing Desk', 'specifications',
     'Electric standing desk with memory presets. Height range: 28-48 inches. Desktop size: 60x30 inches. Weight capacity: 300 lbs. Motor: dual motor system for stability. Includes cable management tray and anti-collision sensor.',
     '2024-01-01'),
    ('Standing Desk', 'assembly',
     'Assembly instructions: 1) Attach legs to frame using provided bolts. 2) Connect motor cables to control box. 3) Mount desktop to frame. 4) Connect power cord. 5) Program height presets using control panel. Assembly time: approximately 45 minutes. Tools needed: Phillips screwdriver.',
     '2024-01-10'),
    ('Office Chair', 'specifications',
     'Ergonomic office chair with lumbar support. Adjustable armrests, seat height, and tilt tension. Breathable mesh back. Seat dimensions: 20x19 inches. Weight capacity: 275 lbs. Warranty: 5 years on frame, 2 years on upholstery.',
     '2024-01-01'),
    ('Office Chair', 'care',
     'Care instructions: Clean mesh with mild soap and water. Lubricate casters annually. Check and tighten bolts every 6 months. Do not exceed weight capacity. Store in dry environment. Replace gas cylinder if chair sinks.',
     '2024-02-01');

### Step 2.5: Verify All Data

In [ ]:
SELECT 'sales' AS table_name, COUNT(*) AS row_count FROM sales
UNION ALL SELECT 'inventory', COUNT(*) FROM inventory
UNION ALL SELECT 'product_docs', COUNT(*) FROM product_docs;

---
## 3. Create Semantic View for Cortex Analyst

**Objective:** Build a semantic model that enables natural language queries against structured data.

A **Semantic View** tells Cortex Analyst:
- Which tables exist and what they contain
- Which columns are **dimensions** (things to group/filter by)
- Which columns are **metrics** (things to measure/aggregate)

This is what allows Cortex Analyst to translate "What are total sales by region?" into correct SQL.

> **Important:** Semantic views use **SQL clause syntax** (`TABLES`, `DIMENSIONS`, `METRICS`), not YAML.

### Step 3.1: Create the Semantic View

We define:
- **TABLES** -- the physical table and its primary key
- **DIMENSIONS** -- `product_name`, `category`, `region`, `customer_segment` (group/filter columns)
- **METRICS** -- `total_sales`, `total_quantity`, `transaction_count` (aggregate measures)

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW sales_semantic_view
  TABLES (
    sales AS CORTEX_AGENTS_LAB.TUTORIAL.SALES
      PRIMARY KEY (sale_id)
      COMMENT = 'Transaction-level sales data'
  )
  DIMENSIONS (
    sales.product_name AS product_name
      COMMENT = 'Name of the product sold',
    sales.category AS category
      COMMENT = 'Product category (Electronics, Furniture)',
    sales.region AS region
      COMMENT = 'Sales region (North America, Europe, Asia Pacific)',
    sales.customer_segment AS customer_segment
      COMMENT = 'Customer type (Enterprise, SMB, Consumer)'
  )
  METRICS (
    sales.total_sales AS SUM(total_amount)
      COMMENT = 'Total sales amount in USD',
    sales.total_quantity AS SUM(quantity)
      COMMENT = 'Total units sold',
    sales.transaction_count AS COUNT(*)
      COMMENT = 'Number of sales transactions'
  )
  COMMENT = 'Sales data for analyzing revenue, products, and regional performance';

### Step 3.2: Verify the Semantic View

In [ ]:
SHOW SEMANTIC VIEWS;

### Step 3.3: Test the Semantic View

We can query the semantic view directly using `SEMANTIC_VIEW()` to confirm the dimensions and metrics work correctly.

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
    CORTEX_AGENTS_LAB.TUTORIAL.SALES_SEMANTIC_VIEW
    METRICS total_sales, total_quantity
    DIMENSIONS region
)
ORDER BY total_sales DESC;

### Step 3.4: Query the Semantic View with Explicit Metrics

The `SEMANTIC_VIEW()` function lets you query the semantic view by specifying **metrics** and **dimensions** directly in SQL. This is useful for ad-hoc analysis where you know exactly what you want to measure and how to slice it.

Try changing the `METRICS` and `DIMENSIONS` below to explore your data in different ways.

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
    CORTEX_AGENTS_LAB.TUTORIAL.SALES_SEMANTIC_VIEW
    METRICS total_sales, transaction_count
    DIMENSIONS region, category
)
ORDER BY total_sales DESC;

---
## 4. Create Cortex Search Service

**Objective:** Index unstructured documents for semantic search (RAG).

A **Cortex Search Service** takes a text column, embeds it into vectors, and enables semantic similarity search. When a user asks a question like "How do I fix laptop overheating?", Cortex Search finds the most relevant documentation chunks.

Key parameters:
- `ON content` -- the text column to index
- `ATTRIBUTES` -- filterable metadata columns (product_name, doc_type)
- `TARGET_LAG` -- how fresh the index should be

### Step 4.1: Create the Search Service

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE product_search_service
  ON content
  ATTRIBUTES product_name, doc_type
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 hour'
  AS (
    SELECT 
      doc_id,
      product_name,
      doc_type,
      content
    FROM product_docs
  );

### Step 4.2: Verify the Search Service

In [ ]:
SHOW CORTEX SEARCH SERVICES;

### Step 4.3: Test the Search Service

Use `SEARCH_PREVIEW` with `FLATTEN` to query the service and verify results come back.

In [ ]:
SELECT 
  r.value:product_name::STRING AS product_name,
  r.value:doc_type::STRING AS doc_type,
  SUBSTRING(r.value:content::STRING, 1, 100) AS content_preview
FROM TABLE(FLATTEN(
  PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
      'CORTEX_AGENTS_LAB.TUTORIAL.PRODUCT_SEARCH_SERVICE',
      '{"query": "laptop overheating fix", "columns": ["content", "product_name", "doc_type"], "limit": 3}'
    )
  )['results']
)) r;

### Step 4.4: RAG -- Answer a Question Using Search + LLM

Step 4.3 retrieved relevant documents, but returned raw text chunks. In a **Retrieval-Augmented Generation (RAG)** pattern, we feed those chunks as context to an LLM so it can synthesize a natural language answer.

The flow:
1. **Retrieve** -- Cortex Search finds the most relevant documents
2. **Augment** -- we inject the retrieved text into a prompt as context
3. **Generate** -- `CORTEX.COMPLETE` produces a grounded answer

This keeps the LLM's response factual because it can only use information from your actual documents.

In [ ]:
-- RAG: Retrieve context from Cortex Search, then generate an answer with an LLM
WITH search_results AS (
    SELECT
        r.value:content::STRING AS content
    FROM TABLE(FLATTEN(
        PARSE_JSON(
            SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
                'CORTEX_AGENTS_LAB.TUTORIAL.PRODUCT_SEARCH_SERVICE',
                '{"query": "How do I fix laptop overheating?", "columns": ["content"], "limit": 3}'
            )
        )['results']
    )) r
),
context AS (
    SELECT LISTAGG(content, '\n\n') AS combined_context
    FROM search_results
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'claude-3-5-sonnet',
    CONCAT(
        'You are a helpful product support assistant. Answer the question using ONLY the context provided below. ',
        'If the context does not contain enough information, say so.\n\n',
        'Context:\n', combined_context, '\n\n',
        'Question: How do I fix laptop overheating?\n\n',
        'Answer:'
    )
) AS rag_answer
FROM context;

---
## 5. Cortex Agent — Unified Natural Language Interface (Optional)

**Objective:** Create a single agent that automatically routes questions to the right tool — Cortex Analyst for structured data queries, or Cortex Search for document retrieval.

Instead of deciding *which* tool to call, you ask a question in plain English and the **Cortex Agent** figures out the best path:

| Question Type | Tool Used | Example |
|---|---|---|
| Structured data / metrics | Cortex Analyst (text-to-SQL) | "What are total sales by region?" |
| Document / knowledge lookup | Cortex Search | "How do I fix laptop overheating?" |
| Both | Agent orchestrates multiple tools | "Show sales for laptops and any known issues" |

> **Note:** Cortex Agents may not be available on all trial accounts. If `CREATE AGENT` or `DATA_AGENT_RUN` returns an error, you can skip this section — the individual Analyst and Search demos above already cover the core functionality.

### Step 5.1: Create the Agent

`CREATE AGENT` defines a reusable agent object with:
- **Tools** — what capabilities the agent has (Analyst for SQL, Search for documents)
- **Tool resources** — which semantic view and search service each tool uses
- **Instructions** — how the agent should behave and route questions
- **Budget** — limits on execution time and tokens

In [ ]:
CREATE OR REPLACE AGENT tutorial_agent
  COMMENT = 'Tutorial agent that routes questions to Analyst (structured) or Search (unstructured)'
  FROM SPECIFICATION $$
models:
  orchestration: claude-4-sonnet
orchestration:
  budget:
    seconds: 30
    tokens: 16000
instructions:
  system: "You are a helpful assistant for a retail business. You can answer questions about sales data and product documentation."
  orchestration: "Use Analyst for any question about sales, revenue, quantities, or metrics. Use Search for product documentation, troubleshooting, or how-to questions."
  response: "Be concise and include relevant numbers or details from the tools."
tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "Analyst"
      description: "Queries structured sales data by converting natural language to SQL"
  - tool_spec:
      type: "cortex_search"
      name: "Search"
      description: "Searches product documentation and troubleshooting guides"
tool_resources:
  Analyst:
    semantic_view: "CORTEX_AGENTS_LAB.TUTORIAL.SALES_SEMANTIC_VIEW"
    execution_environment:
      type: "warehouse"
      warehouse: "COMPUTE_WH"
  Search:
    name: "CORTEX_AGENTS_LAB.TUTORIAL.PRODUCT_SEARCH_SERVICE"
    max_results: "3"
$$;

### Step 5.2: Ask a Structured Data Question

Use `DATA_AGENT_RUN` to send a natural language question to the agent. The agent recognizes this as a **structured data question** and routes it to **Cortex Analyst**, which generates and executes SQL against the semantic view.

The response is a JSON string. We parse it in Python to display:
- **Thinking** — the agent's reasoning about which tool to use
- **Generated SQL** — the query Cortex Analyst produced
- **Result set** — the data returned by the query
- **Answer** — the agent's final natural language response

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Run the agent with a structured data question
result = session.sql("""
  SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'CORTEX_AGENTS_LAB.TUTORIAL.TUTORIAL_AGENT',
    $${"messages": [{"role": "user", "content": [{"type": "text", "text": "What are total sales by region?"}]}]}$$
  ) AS resp
""").collect()

resp = json.loads(result[0]["RESP"])

# Parse and display each content item
for item in resp.get("content", []):
    item_type = item.get("type")

    if item_type == "thinking":
        print("=== THINKING ===")
        print(item["thinking"]["text"])
        print()

    elif item_type == "tool_use":
        print(f"=== TOOL CALL: {item['tool_use'].get('name', '')} ({item['tool_use'].get('type', '')}) ===")
        print()

    elif item_type == "tool_result":
        tr = item["tool_result"]
        print(f"=== TOOL RESULT: {tr.get('name', '')} ===")
        content_json = tr.get("content", [{}])[0].get("json", {})
        if content_json.get("sql"):
            print(f"Generated SQL:\n{content_json['sql']}\n")
        if content_json.get("sql_explanation"):
            print(f"Explanation: {content_json['sql_explanation']}\n")
        if content_json.get("result_set", {}).get("data"):
            meta = content_json["result_set"].get("resultSetMetaData", {})
            cols = [col["name"] for col in meta.get("rowType", [])]
            data = content_json["result_set"]["data"]
            if cols:
                print(" | ".join(cols))
                print("-" * (len(" | ".join(cols))))
            for row in data:
                print(" | ".join(str(v) for v in row))
            print()

    elif item_type == "text":
        print("=== ANSWER ===")
        print(item["text"])
        print()

### Step 5.3: Ask a Documentation Question

Now ask the **same agent** a question about product documentation. The agent recognizes this as an **unstructured data question** and routes it to **Cortex Search** to find relevant documents.

This is the key value of Cortex Agents — one interface, automatic routing to the right tool.

In [ ]:
# Run the agent with a documentation question
result = session.sql("""
  SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'CORTEX_AGENTS_LAB.TUTORIAL.TUTORIAL_AGENT',
    $${"messages": [{"role": "user", "content": [{"type": "text", "text": "How do I fix laptop overheating?"}]}]}$$
  ) AS resp
""").collect()

resp = json.loads(result[0]["RESP"])

for item in resp.get("content", []):
    item_type = item.get("type")

    if item_type == "thinking":
        print("=== THINKING ===")
        print(item["thinking"]["text"])
        print()

    elif item_type == "tool_use":
        print(f"=== TOOL CALL: {item['tool_use'].get('name', '')} ({item['tool_use'].get('type', '')}) ===")
        print()

    elif item_type == "tool_result":
        tr = item["tool_result"]
        print(f"=== TOOL RESULT: {tr.get('name', '')} (status: {tr.get('status', '')}) ===")
        print()

    elif item_type == "text":
        print("=== ANSWER ===")
        print(item["text"])
        print()

---
## 6. Use with Snowflake Intelligence

Now that you've created the `tutorial_agent` object, you can chat with it directly in **Snowflake Intelligence** — no extra code needed.

**To try it:**
1. In Snowsight, go to **AI & ML > Snowflake Intelligence**
2. Select `TUTORIAL_AGENT` from the agent picker in the chat bar
3. Ask any question — the agent routes to Analyst or Search automatically
   - Try **"What are total sales by region?"** (routes to Cortex Analyst)
   - Try **"How do I assemble the standing desk?"** (routes to Cortex Search)

Snowflake Intelligence gives you a built-in chat interface with:
- **Streaming responses** — see the agent's thinking and tool calls in real time
- **Multi-turn conversations** — ask follow-up questions that build on previous context
- **Tool call visibility** — see which tool the agent chose and the generated SQL or search results

This is the fastest way to interact with the agent you just built. For a custom UI, you can build a Streamlit app on top of the same agent using the `DATA_AGENT_RUN` SQL function or the Cortex Agents REST API.

---
## Summary

You've built a complete natural language query pipeline with Snowflake Cortex:

| Component | Object | What It Enables |
|-----------|--------|----------------|
| Structured data | `sales`, `inventory` | Transaction and stock data for SQL queries |
| Unstructured data | `product_docs` | Product documentation for semantic retrieval |
| **Semantic View** | `sales_semantic_view` | Cortex Analyst translates plain English into SQL |
| **Search Service** | `product_search_service` | Cortex Search finds relevant documents via similarity |
| **Cortex Agent** | `tutorial_agent` | Unified interface that routes questions to the right tool |

**Where to go from here:**
- **Streamlit app** -- build an interactive chat interface on top of the agent
- **Expand the semantic view** -- add more tables, dimensions, and metrics as your data grows
- **Add more documents** -- index additional content into the search service for broader coverage
- **Add more tools** -- give the agent access to `sql_exec`, `data_to_chart`, or custom functions

---
## Cleanup (Optional)

Uncomment and run to remove all tutorial objects.

In [ ]:
-- DROP AGENT IF EXISTS tutorial_agent;
-- DROP CORTEX SEARCH SERVICE IF EXISTS product_search_service;
-- DROP SEMANTIC VIEW IF EXISTS sales_semantic_view;
-- DROP TABLE IF EXISTS sales;
-- DROP TABLE IF EXISTS inventory;
-- DROP TABLE IF EXISTS product_docs;